# District 1: Missing RT https://github.com/cal-itp/gtfs-curator/issues/82

In [49]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

# from shared_utils import bq_utils
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data
from update_vars import DIGEST_DICT

In [2]:
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, abbrev_month, analysis_month 

In [3]:
@cache
def gcs_pandas():
    return GCSPandas()

In [4]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [5]:
agencies_with_rt = ["Humboldt Transit Authority",
                   "Lake Transit Authority",
                   "Redwood Coast Transit Authority"]

In [6]:
raw_url = "gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_08.parquet"

In [7]:
fct_monthly_route_df = pd.read_parquet(raw_url,
                                        storage_options={"token": credentials.token}).reset_index()

In [8]:
fct_monthly_route_df.sample()

,index,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
2062,2062,8,2026,2026-08-01,B-Line Schedule,aHR0cHM6Ly9wbGFubXlyaWRlLmJsaW5ldHJhbnNpdC5jb20vZ3Rmc3JlYWx0aW1lL2dvb2dsZV90cmFuc2l0LnppcA==,B-Line Vehicle Positions,aHR0cHM6Ly9wbGFubXlyaWRlLmJsaW5ldHJhbnNpdC5jb20vZ3Rmc3JlYWx0aW1lL0dURlNfVmVoaWNsZVBvc2l0aW9ucy5wYg==,B-Line Trip Updates,aHR0cHM6Ly9wbGFubXlyaWRlLmJsaW5ldHJhbnNpdC5jb20vZ3Rmc3JlYWx0aW1lL0dURlNfVHJpcFVwZGF0ZXMucGI=,Sunday,50,10.00,45.90,9.20,0.00,0.00,1,2,55,1550,310.00,5,2,3.00,50,10.00,1.00,1.00,598.20,2.90,50,10.00,1.00,1.00,1764.40,1.00,0.75,0.20,61456.00,0.51,0.03,0.47,0.00,5.00,0.00,0.00,"[-230.3, -164.0, -104.0, -80.3, -58.0, -19.0, 3.3, 29.0, 63.7, 82.0, 102.0, 216.0, 356.0]","[5, 10, 20, 25, 30, 40, 50, 60, 70, 75, 80, 90, 95]","[-0.477, -0.267, -0.127, -0.098, -0.075, -0.028, 0.004, 0.042, 0.081, 0.111, 0.155, 0.295, 0.44]","[7.0, 13.0, 26.7, 36.0, 44.0, 61.0, 80.0, 100.0, 139.3, 165.0, 213.0, 355.0, 451.7]","[-400.0, -241.3, -171.0, -153.7, -137.7, -110.0, -87.0, -68.7, -47.0, -36.7, -29.3, -14.7, -6.7]","[95, 90, 80, 75, 70, 60, 50, 40, 30, 25, 20, 10, 5]"


In [15]:
fct_monthly_route_df.schedule_name.unique()

array(['Yuba-Sutter Schedule', 'Petaluma GMV Schedule',
       'Anteater Express Schedule', 'MVGO Schedule', 'Calaveras Schedule',
       'Lake Schedule', 'Artesia Schedule', 'YARTS Schedule',
       'Bay Area 511 Emery Express Schedule', 'ACE Schedule',
       'Roseville Transit TripShot Schedule',
       'Bay Area 511 Fairfield and Suisun Transit Schedule',
       'Lassen Schedule', 'CSULB Shuttle PassioGo Schedule',
       'Clovis Schedule', 'WestCAT Flex', 'Wasco DAR Schedule',
       'Curry Public Transit Schedule', 'Alhambra Schedule',
       'Glendora Schedule', 'San Juan Capistrano Trolley Schedule',
       'La Puente Schedule', 'Bay Area 511 MVGO Schedule',
       'Eastern Sierra Flex', 'Moorpark Schedule', 'Delano Schedule',
       'Madera Metro Schedule', 'Trinity Remix Schedule',
       'Bellflower Remix Schedule', 'El Segundo Schedule',
       'Rio Vista Schedule', 'Tri-Valley Wheels Schedule',
       'Thousand Oaks Flex', 'Compton Schedule', 'Turlock Flex',
       'Calaba

In [9]:
fct_monthly_route_df2 = fct_monthly_route_df.loc[fct_monthly_route_df.schedule_name.isin(
 ["Lake Schedule"] 
)]

In [10]:
fct_monthly_route_df2.sample()

,index,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
5,5,8,2026,2026-08-01,Lake Schedule,None,None,None,None,None,Sunday,0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.00,5,1,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [11]:
fct_monthly_route_df2.filter(like="vp").columns

Index(['vp_name', 'vp_base64_url', 'vp_messages_per_minute', 'n_vp_trips',
       'daily_vp_trips', 'pct_vp_trips', 'pct_vp_routes',
       'daily_vp_extract_duration_minutes', 'n_days_schedule_and_vp_only'],
      dtype='object')

In [12]:
fct_monthly_route_df2.pct_vp_trips.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: pct_vp_trips, dtype: float64

In [13]:
fct_monthly_route_df2.pct_vp_routes.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: pct_vp_routes, dtype: float64

In [14]:
fct_monthly_route_df2.vp_messages_per_minute.describe()

count   0.00
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: vp_messages_per_minute, dtype: float64

# Also noticed a lot of AC Transit RT is missing
* My code is dropping the previous months with data.

##  VP and TU Missing

In [20]:
operator_summary_url = "gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_08.parquet"

In [22]:
operator_df = gcs_pandas().read_parquet(
    operator_summary_url)

In [23]:
operator_df.sample()

,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
601,9,2026,2026-09-01,LA Metro Events Schedule,None,None,None,None,None,Weekday,0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.00,3,2,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [25]:
ac_transit = operator_df.loc[operator_df.schedule_name == "Bay Area 511 AC Transit Schedule"]

In [29]:
ac_transit.sample()

,month,year,month_first_day,schedule_name,schedule_base64_url,vp_name,vp_base64_url,tu_name,tu_base64_url,day_type,n_trips,daily_trips,ttl_service_hours,daily_service_hours,ttl_flex_service_hours,daily_flex_service_hours,n_routes,n_shapes,n_stops,num_stop_times,daily_arrivals,n_days,n_feeds,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,pct_vp_routes,daily_vp_extract_duration_minutes,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips,pct_tu_routes,daily_tu_extract_duration_minutes,pct_tu_complete_minutes,pct_tu_accurate_minutes,avg_prediction_spread_minutes,n_predictions,pct_predictions_early,pct_predictions_ontime,pct_predictions_late,n_days_schedule_only,n_days_schedule_and_rt,n_days_schedule_and_vp_only,n_days_schedule_and_tu_only,prediction_error_sec_array,prediction_error_sec_percentile_array,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array
512,8,2026,2026-08-01,Bay Area 511 AC Transit Schedule,None,None,None,None,None,Weekday,0,0.00,0.00,0.00,0.00,0.00,0,0,0,0,0.00,1,1,NaN,0,0.00,NaN,NaN,NaN,NaN,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],[],[],[],[],[]


In [31]:
ac_transit2 = ac_transit.dropna(subset = ["n_vp_trips","pct_vp_trips","pct_tu_trips","pct_tu_routes"])

In [30]:
ac_transit[["day_type","month_first_day","n_vp_trips","pct_vp_trips","pct_tu_trips","pct_tu_routes"]].sort_values(by = ["month_first_day"])

,day_type,month_first_day,n_vp_trips,pct_vp_trips,pct_tu_trips,pct_tu_routes
1488,Saturday,2026-07-01,14193,0.89,0.98,1.00
1489,Sunday,2026-07-01,12110,0.76,0.91,1.00
1490,Weekday,2026-07-01,104132,0.92,0.99,1.00
509,Saturday,2026-08-01,0,NaN,NaN,NaN
512,Weekday,2026-08-01,0,NaN,NaN,NaN
1486,Saturday,2026-08-01,14376,0.90,0.98,1.00
1487,Sunday,2026-08-01,16875,0.84,0.96,1.00
1491,Weekday,2026-08-01,93302,0.92,0.98,1.00
1485,Weekday,2026-09-01,10016,0.92,0.99,1.00


In [34]:
GCS_PATH = f"{GTFS_DATA_DICT.dir}processed/"

In [35]:
f"{GCS_PATH}{GTFS_DATA_DICT.schedule_rt_route_direction}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_schedule_rt_route_direction_summary.parquet'

In [36]:
ac_transit2 = ac_transit2.loc[ac_transit2.day_type == "Weekday"]

In [37]:
ac_transit2[["day_type","month_first_day","n_vp_trips","pct_vp_trips","pct_tu_trips","pct_tu_routes"]].sort_values(by = ["month_first_day"])

,day_type,month_first_day,n_vp_trips,pct_vp_trips,pct_tu_trips,pct_tu_routes
1490,Weekday,2026-07-01,104132,0.92,0.99,1.00
1491,Weekday,2026-08-01,93302,0.92,0.98,1.00
1485,Weekday,2026-09-01,10016,0.92,0.99,1.00


In [ ]:
operator_df = gcs_pandas().read_parquet(
    operator_summary_url,
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "==", analysis_name)],
).drop_duplicates(subset = ["Analysis Name", "Date"])

In [39]:
import _new_operator_report_utils as utils

## Scheduled Route Direction Summary

In [52]:
"gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_schedule_rt_route_direction_summary_2026_08.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_schedule_rt_route_direction_summary_2026_08.parquet'

In [53]:
DIGEST_DICT.schedule_rt_route_direction

'fct_monthly_schedule_rt_route_direction_summary'

In [60]:
df = gcs_pandas().read_parquet("gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_schedule_rt_route_direction_summary_2026_08.parquet"
    )

In [61]:
df.columns

Index(['month', 'year', 'month_first_day', 'day_type', 'schedule_base64_url',
       'name', 'tu_base64_url', 'tu_name', 'vp_base64_url', 'vp_name',
       'route_name', 'direction_id', 'route_type', 'n_trips',
       'daily_trips_all_day', 'n_route_ids', 'n_shapes', 'avg_stops_served',
       'ttl_service_hours', 'ttl_flex_service_hours', 'daily_service_hours',
       'daily_flex_service_hours', 'n_feeds', 'n_days', 'daily_trips_owl',
       'daily_trips_early_am', 'daily_trips_am_peak', 'daily_trips_midday',
       'daily_trips_pm_peak', 'daily_trips_evening', 'daily_trips_peak',
       'daily_trips_offpeak', 'frequency_owl', 'frequency_early_am',
       'frequency_am_peak', 'frequency_midday', 'frequency_pm_peak',
       'frequency_evening', 'frequency_peak', 'frequency_offpeak',
       'frequency_all_day', 'appeared_in_tu', 'appeared_in_vp',
       'vp_num_distinct_updates', 'n_vp_trips', 'vp_extract_duration_minutes',
       'vp_messages_per_minute', 'tu_num_distinct_updates', 'n_

In [62]:
schedule_route_dir_columns_to_keep = [
                "month_first_day",
                "name",
                "route_name",
                "direction_id",
                "frequency_all_day",
                "frequency_offpeak",
                "frequency_peak",
                "daily_trips_peak",
                "daily_trips_offpeak",
                "daily_trips_all_day",
                "day_type",
                "route_type",
                "daily_service_hours",
                "route_typology",
            ]

In [63]:
df = df[schedule_route_dir_columns_to_keep]

In [65]:
df = df.loc[df.name.isin(['Bay Area 511 AC Transit Schedule',
       'AC Transit Schedule'])]

In [66]:
df.shape

(8990, 14)

In [69]:
df[["month_first_day","route_name","frequency_all_day",
                "frequency_offpeak",
                "frequency_peak","daily_service_hours",
   ]].sort_values(by = ["month_first_day","route_name"], 
                  ascending =  [False, False])

,month_first_day,route_name,frequency_all_day,frequency_offpeak,frequency_peak,daily_service_hours
18811,2026-09-01,W__W Bay Farm - South Shore Transbay,0.00,0.13,0.13,2.83
18825,2026-09-01,W__W Bay Farm - South Shore Transbay,0.00,0.13,0.13,2.83
55441,2026-09-01,W__W Bay Farm - South Shore Transbay,0.00,0.00,0.75,6.12
55455,2026-09-01,W__W Bay Farm - South Shore Transbay,0.00,0.00,0.75,6.12
34244,2026-09-01,V__V Montclair - Park Blvd. Transbay,0.00,0.13,0.38,3.87
34248,2026-09-01,V__V Montclair - Park Blvd. Transbay,0.00,0.13,0.38,3.87
47770,2026-09-01,V__V Montclair - Park Blvd. Transbay,0.00,0.00,0.63,4.55
47783,2026-09-01,V__V Montclair - Park Blvd. Transbay,0.00,0.00,0.63,4.55
34179,2026-09-01,U__U Stanford - Dumbarton - Fremont,0.00,0.13,0.38,6.27
34184,2026-09-01,U__U Stanford - Dumbarton - Fremont,0.00,0.13,0.38,6.27
